# 4 · The lowest-effort path

Minimal cut sets, three meanings of "lowest effort", importance, and engineering out. See
[Lesson 4](../docs/lessons/04-the-lowest-effort-path.md).

In [1]:
import matplotlib.pyplot as plt

from cpsrbd import plotting

plotting.use_style()

In [2]:
from cpsrbd.consequence_driven_analysis import navigation_case_study
from cpsrbd.critical_path_identification import (
    activated,
    cheapest_attacks,
    importance_measures,
    min_attacker_cost,
    most_likely,
    opportunistic_front,
)

model = navigation_case_study()
for c in most_likely(model):
    print(f"{c.probability:.2e}  {c.kind:8s}  {c.label()}")

5.00e-04  physical  mc_hw
3.16e-04  mixed     gnss_spoof + integrity_monitor_miss
1.33e-04  cyber     compromise_maintenance_laptop + flood_bus
5.62e-05  cyber     compromise_maintenance_laptop + inject_false_nav_messages
4.22e-06  cyber     flood_bus + supply_chain_implant
3.99e-06  physical  egi_1_hw + egi_2_hw
1.78e-06  cyber     inject_false_nav_messages + supply_chain_implant
4.00e-07  physical  egi_1_hw + power_bus_2
4.00e-07  physical  egi_2_hw + power_bus_1
4.00e-08  physical  power_bus_1 + power_bus_2
1.33e-08  cyber     compromise_maintenance_laptop + defeat_load_verification + malicious_program_download
1.00e-08  physical  bus_ch_a + bus_ch_b


In [3]:
print("cheapest attack:", cheapest_attacks(model, 1)[0].label(), f"({min_attacker_cost(model):g} weeks)")
print("opportunistic front:")
for c in opportunistic_front(model):
    print(f"  cost {c.attacker_cost:4g}  P(physical preconditions) {c.physical_probability:.1e}  {c.label()}")

cheapest attack: compromise_maintenance_laptop + flood_bus (5 weeks)
opportunistic front:
  cost    0  P(physical preconditions) 5.0e-04  mc_hw
  cost    4  P(physical preconditions) 1.0e-01  gnss_spoof + integrity_monitor_miss
  cost    5  P(physical preconditions) 1.0e+00  compromise_maintenance_laptop + flood_bus


In [4]:
best = cheapest_attacks(model, 1)[0]
fig = plotting.draw_model(model, highlight=activated(model, best.events), title=f"Cheapest attack: {best.label()}")

Importance, and the risk left after engineering out each cyber step:

In [5]:
for r in importance_measures(model)[:8]:
    print(f"{r.event:30s} FV {r.fussell_vesely:.3f}  RAW {r.raw:7.1f}  Q without {r.q_without:.3e}")

mc_hw                          FV 0.492  RAW   985.1  Q without 5.155e-04
gnss_spoof                     FV 0.312  RAW    99.1  Q without 6.991e-04
integrity_monitor_miss         FV 0.312  RAW     3.8  Q without 6.991e-04
compromise_maintenance_laptop  FV 0.186  RAW    25.6  Q without 8.267e-04
flood_bus                      FV 0.135  RAW     8.4  Q without 8.787e-04
inject_false_nav_messages      FV 0.057  RAW     8.4  Q without 9.582e-04
supply_chain_implant           FV 0.006  RAW    25.6  Q without 1.009e-03
egi_1_hw                       FV 0.004  RAW     3.2  Q without 1.011e-03
